In [1]:
import os
os.environ['TORCH_HOME'] = '/project/dl2023s/hoos/dl-lab-final-project/cache'
%cd /project/dl2023s/hoos/dl-lab-final-project

/project/dl2023s/hoos/dl-lab-final-project


In COCO.loadRes(res_file):
- only images that exists in the res_file will be used for evaluation
- an additional field "id" is added to each annotation in res
- return res (a COCO instance)

Format of res:
[ {"image_id": int, "caption": str} ]

In [2]:
import json
import pickle

refcoco_plus_instance_file = "./datasets/refcoco+/instances.json"
refcoco_plus_ref_file = "./datasets/refcoco+/refs(unc).p"
refcoco_instances = json.load(open(refcoco_plus_instance_file, "r"))
refcoco_ref = pickle.load(open(refcoco_plus_ref_file, "rb"))

In [3]:
refcoco_ref[0]

{'sent_ids': [0, 1, 2],
 'file_name': 'COCO_train2014_000000581857_16.jpg',
 'ann_id': 1719310,
 'ref_id': 0,
 'image_id': 581857,
 'split': 'train',
 'sentences': [{'tokens': ['navy', 'blue', 'shirt'],
   'raw': 'navy blue shirt',
   'sent_id': 0,
   'sent': 'navy blue shirt'},
  {'tokens': ['woman', 'back', 'in', 'blue'],
   'raw': 'woman back in blue',
   'sent_id': 1,
   'sent': 'woman back in blue'},
  {'tokens': ['blue', 'shirt'],
   'raw': 'blue shirt',
   'sent_id': 2,
   'sent': 'blue shirt'}],
 'category_id': 1}

In [21]:
class COCOEvalCaptionByAnnotationId:
    def __init__(self, coco_gt, coco_res):
        '''
        Initialize CocoEval using coco APIs for gt and dt
        :param coco_gt: coco object with ground truth annotations
        :param coco_res: coco object with detection 
        '''
        self.eval_ann_ids = []
        self.eval = {}
        self.ann_to_eval = {}


    def evaluate(self):
        # assume ground truth and results contain the same set of anns
        ann_ids_to_eval = ...

        gts = {}
        res = {}
        for ann_id in ann_ids_to_eval:
            gts[ann_id] = ... #[ {"ann_id": int, "caption": str}, ... ]
            res[ann_id] = ... #[ {"ann_id": int, "caption": str}, ... ]


[{'image_id': 179765,
  'id': 38,
  'caption': 'A black Honda motorcycle parked in front of a garage.'},
 {'image_id': 179765,
  'id': 182,
  'caption': 'A Honda motorcycle parked in a grass driveway'},
 {'image_id': 190236,
  'id': 401,
  'caption': 'An office cubicle with four different types of computers.'},
 {'image_id': 331352,
  'id': 441,
  'caption': 'A small closed toilet in a cramped space.'},
 {'image_id': 517069,
  'id': 447,
  'caption': 'Two women waiting at a bench next to a street.'},
 {'image_id': 179765,
  'id': 479,
  'caption': 'A black Honda motorcycle with a dark burgundy seat.'},
 {'image_id': 331352,
  'id': 540,
  'caption': 'A tan toilet and sink combination in a small room.'},
 {'image_id': 190236,
  'id': 644,
  'caption': 'The home office space seems to be very cluttered.'},
 {'image_id': 182417,
  'id': 856,
  'caption': 'A beautiful dessert waiting to be shared by two people'},
 {'image_id': 517069,
  'id': 882,
  'caption': 'A woman sitting on a bench an

In [38]:
d_gts = {}
d_gts[182417] = coco_2017.imgToAnns[182417]
print(d_gts)

{182417: [{'image_id': 182417, 'id': 856, 'caption': 'A beautiful dessert waiting to be shared by two people'}, {'image_id': 182417, 'id': 4882, 'caption': 'There is a piece of cake on a plate with decorations on it.'}, {'image_id': 182417, 'id': 5098, 'caption': 'Creamy cheesecake dessert with whip cream and caramel.'}, {'image_id': 182417, 'id': 10897, 'caption': 'An extravagant dessert on a plate overlooking the water.'}, {'image_id': 182417, 'id': 824547, 'caption': 'This is a picture of an extremely fancy desert.'}]}


In [39]:
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

tokenizer = PTBTokenizer()
d_gts = tokenizer.tokenize(d_gts)

PTBTokenizer tokenized 57 tokens at 237.31 tokens per second.


{
    "image": [{"id": ann_id}, ...],
    "annotations": [ {"image_id": ann_id, "caption": str}, ... ],
}

In [3]:
ann_id_to_captions = {}
for ref in refcoco_ref:
    ref_sentences = ref['sentences']
    sentences = []
    for sent in ref_sentences:
        sentences.append(sent['raw'])
    ann_id_to_captions[ref['ann_id']] = sentences

ann_id_to_captions

{1719310: ['navy blue shirt', 'woman back in blue', 'blue shirt'],
 463958: ['grey shirt wearing glasses',
  'lady with glasses',
  'woman with grey shirt standing next to man'],
 495152: ['standing with racket', 'woman standing', 'holding raquet'],
 485695: ['woman sitting with leg extended', 'ditting darker hair'],
 453177: ['child with case on head',
  'holding a green object',
  'hand of person with red shirt'],
 440706: ['guy in brown shirt', 'man in brown', 'man wearing long sleeves'],
 1097431: ['urinal closest to the white pipe'],
 1093269: ['pisser far Rat'],
 1154905: ['vase holding tallest flower',
  'The vase with the tallest flower.',
  'no 3'],
 1152539: ['vase of flower drooping',
  'vase with wilted flower',
  'vase of the flower tilting over'],
 1151495: ['next to central area on side of hand least used by most people',
  'vase touching drooping flower'],
 247896: ["red bus you don't see the front of",
  'the red bus thats blocked in',
  'Red bus with windshield hidden

In [41]:
modified_refcoco = { "images": [], "annotations": [] }
for ann_id in ann_id_to_captions.keys():
    modified_refcoco["images"].append({ "id": ann_id })
    for caption in ann_id_to_captions[ann_id]:
        modified_refcoco["annotations"].append({ "image_id": ann_id, "caption": caption, "id": ann_id })

modified_refcoco

{'images': [{'id': 1719310},
  {'id': 463958},
  {'id': 495152},
  {'id': 485695},
  {'id': 453177},
  {'id': 440706},
  {'id': 1097431},
  {'id': 1093269},
  {'id': 1154905},
  {'id': 1152539},
  {'id': 1151495},
  {'id': 247896},
  {'id': 165092},
  {'id': 2204728},
  {'id': 516523},
  {'id': 511746},
  {'id': 41223},
  {'id': 38565},
  {'id': 1103476},
  {'id': 1101817},
  {'id': 556799},
  {'id': 1758997},
  {'id': 2157900},
  {'id': 508442},
  {'id': 500335},
  {'id': 1345868},
  {'id': 344837},
  {'id': 1082811},
  {'id': 1081593},
  {'id': 2218461},
  {'id': 309123},
  {'id': 71392},
  {'id': 70901},
  {'id': 510191},
  {'id': 505664},
  {'id': 2005581},
  {'id': 531973},
  {'id': 490700},
  {'id': 1991154},
  {'id': 1991009},
  {'id': 336962},
  {'id': 57804},
  {'id': 55711},
  {'id': 591091},
  {'id': 588533},
  {'id': 1313895},
  {'id': 459894},
  {'id': 1816401},
  {'id': 48922},
  {'id': 538268},
  {'id': 534500},
  {'id': 1074443},
  {'id': 1073591},
  {'id': 464938},
  {

In [40]:
# res = [
#     {"image_id": 1103476, "caption": "dummy caption"},
#     {"image_id": 430836, "caption": "dummy caption 123"},
# ]

res = [
    {"image_id": 1151495, "caption": "next to central area on side of hand least used by most people"},
    {"image_id": 247896, "caption": "red bus you don't see the front of"},
]

In [42]:
def filter_coco_gt(coco_gt, img_ids):
    # create dictionaries of images and annotation with image_id as key
    images = {}
    for img in coco_gt["images"]:
        images[img["id"]] = img

    annotations = {}
    for ann in coco_gt["annotations"]:
        if annotations.get(ann['image_id']) is None:
            annotations[ann['image_id']] = [ann]
        else:
            annotations[ann['image_id']].append(ann)

    # filter images and annotations
    coco_filter_gt = {"images": [], "annotations": []}
    seen_images = set()
    for img_id in img_ids:
        if img_id in seen_images:
            continue

        coco_filter_gt["images"].append(images[img_id])
        seen_images.add(img_id)

        for ann in annotations[img_id]:
            coco_filter_gt["annotations"].append(ann)

    print("raw coco_gt size: ", len(coco_gt["images"]))
    print("filtered coco_gt size: ", len(coco_filter_gt["images"]))
          
    return coco_filter_gt

relevant_id = [image['image_id'] for image in res]
modified_refcoco = filter_coco_gt(modified_refcoco, relevant_id)

raw coco_gt size:  49856
filtered coco_gt size:  2


In [43]:
import json

# save modified refcoco to a json file
modified_refcoco_file = "./datasets/refcoco+/modified_refcoco.json"
with open(modified_refcoco_file, "w") as f:
    json.dump(modified_refcoco, f)

In [44]:
from pycocotools.coco import COCO

refcoco = COCO(modified_refcoco_file)

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


In [45]:
refcoco_res = refcoco.loadRes(res)

Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!


In [46]:
%load_ext autoreload
%autoreload 2

from pycocoevalcap.eval import COCOEvalCap

coco_eval = COCOEvalCap(refcoco, refcoco_res)
coco_eval.evaluate()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
tokenization...


PTBTokenizer tokenized 41 tokens at 304.48 tokens per second.
PTBTokenizer tokenized 23 tokens at 198.89 tokens per second.


setting up scorers...
computing Bleu score...
bleu hypo ['next to central area on side of hand least used by most people']
bleu ref ['next to central area on side of hand least used by most people', 'vase touching drooping flower']
bleu hypo ["red bus you do n't see the front of"]
bleu ref ["red bus you do n't see the front of", 'the red bus thats blocked in', 'red bus with windshield hidden']
{'testlen': 22, 'reflen': 22, 'guess': [22, 20, 18, 16], 'correct': [22, 20, 18, 16]}
ratio: 0.9999999999545455
Bleu_1: 1.000
Bleu_2: 1.000
Bleu_3: 1.000
Bleu_4: 1.000
computing METEOR score...
METEOR: 1.000
computing Rouge score...
ROUGE_L: 1.000
computing CIDEr score...
CIDEr: 4.549
computing SPICE score...


Parsing reference captions
Initiating Stanford parsing pipeline
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator tokenize
[main] INFO edu.stanford.nlp.pipeline.TokenizerAnnotator - TokenizerAnnotator: No tokenizer type provided. Defaulting to PTBTokenizer.
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator ssplit
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator parse
[main] INFO edu.stanford.nlp.parser.common.ParserGrammar - Loading parser from serialized file edu/stanford/nlp/models/lexparser/englishPCFG.ser.gz ... 
done [0.7 sec].
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator lemma
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator ner
Loading classifier from edu/stanford/nlp/models/ner/english.all.3class.distsim.crf.ser.gz ... done [1.8 sec].
Loading classifier from edu/stanford/nlp/models/ner/english.muc.7class.distsim.crf.ser.gz ... done [1.0 sec].
Loading classif

SPICE evaluation took: 9.845 s
SPICE: 0.686
